# Pseudonimizacja - jak nie wyciec danych do OpenAI?

## Cel: Zbudowanie "firewalla prywatności", który pozwoli bezpiecznie przetwarzać wrażliwe dane w chmurze

### Czego się nauczysz:
1. Zrozumiesz dlaczego większość firm nie potrzebuje własnej infrastruktury do modeli językowych
2. Poznasz mechanizm pseudo-anonimizacji pozwalający bezpiecznie korzystać z modeli zamkniętych
3. Nauczysz się łączyć modele NLP działające na CPU z potężnymi modelami w chmurze
4. Dowiesz się jak pisać własne reguły wykrywania danych dla polskiej specyfiki językowej
5. Zrozumiesz różnicę w nakładach pracy między utrzymaniem modelu lokalnego a użyciem API
6. Zbudujesz kompletny potok przetwarzania ukrywający dane wrażliwe przed wysyłką
7. Otrzymasz gotowe wzorce projektowe do zarządzania stanem konwersacji przy ukrytych danych


### Zwróć uwagę
Pseudonimizacja (zgodnie z [Art. 4](https://gdpr.pl/baza-wiedzy/akty-prawne/interaktywny-tekst-gdpr/artykul-4-definicje) pkt 5 RODO) polega na przetworzeniu danych osobowych w taki sposób, by nie można ich było przypisać konkretnej osobie bez użycia dodatkowych informacji (klucza), które przechowujemy osobno.

## Dylemat inżyniera, czyli między młotem a kowadłem

Zanim zaczniemy pisać kod, musimy ustalić pewne fakty. Brutalna prawda o pracy z danymi w środowisku komercyjnym jest taka, że nie ma rozwiązań idealnych – są tylko kompromisy.

Gdy budujesz system dla banku, szpitala czy kancelarii prawnej, stajesz przed wyborem architektury, który definiuje bezpieczeństwo Twoich danych. Spójrzmy na to spektrum "na chłodno".

### 1. Zamknięte modele (OpenAI, Anthropic, Google)
Tutaj sprawa jest bardziej zniuansowana, niż się wydaje. To nie jest po prostu "wysyłam lub nie". Mamy tu trzy poziomy bezpieczeństwa (i obietnic):
- **Poziom Free (ChatGPT):** Najniższy standard. Istnieje duża szansa, że dane, które tam wkleisz, zostaną użyte do treningu kolejnych wersji modelu. Z inżynierskiego punktu widzenia: traktuj to jak publiczny post na forum.
- **Poziom API (Paid):** Tutaj dostawcy (OpenAI, Anthropic) deklarują, że dane przesyłane przez API **nie są** używane do treningu i są przechowywane tylko przez krótki okres (retencja 30 dni do celów monitorowania abuse). To standard rynkowy.
- **Poziom Enterprise (np. Azure OpenAI):** Dedykowane instancje z umowami SLA i certyfikatami zgodności. Tutaj Microsoft daje gwarancje prawne i separację danych. To najbezpieczniejsza opcja w chmurze publicznej.

### 2. Iluzja "bezpiecznych modeli lokalnych"
W tym momencie często pojawia się pokusa: *"Uruchommy to lokalnie! Będzie taniej i bezpieczniej, bo dane nie opuszczą firmy"*.

Tu jest pewna pułapka myślowa. Dlaczego? Pomyślmy o tym:
- **Bezpieczeństwo fizyczne:** Paradoksalnie, serwerownia Azure z certyfikatami ISO, strażnikami i redundancją jest często bezpieczniejsza (w świetle RODO) niż "karta graficzna pod biurkiem w biurze", do której dostęp ma sprzątaczka, a dysk nie jest nawet zaszyfrowany.
- **Mit taniego hostingu:** Wydaje się, że własny model jest tańszy. To bzdura. W OpenAI pracują setki inżynierów, których jedynym celem jest optymalizacja inferencji (batching, kv-caching, kwantyzacja itd). Szansa, że Twój zespół DevOps "z doskoku" uruchomi model Llama/DeepSeek wydajniej i taniej niż oni, jest bliska zeru.
- **Dostawcy Cloud GPU:** Jeśli nie masz własnego sprzętu i wynajmujesz GPU (np. [RunPod](https://www.runpod.io), [Lambda Labs](https://lambda.ai)), to Twoje dane i tak wędrują na cudzy serwer. Różnica jest taka, że zamiast do Microsoftu, wysyłasz je do mniejszej firmy, która może mieć znacznie słabsze procedury bezpieczeństwa.

### 3. Inżynierski kompromis: Pseudo-anonimizacja

Skoro modele lokalne są trudne w utrzymaniu i drogie, a modele chmurowe (Enterprise) są genialne, ale wymagają wysyłania danych, to czy istnieje trzecia droga?

Tak. Możemy wykorzystać inteligencję modeli SOTA (State of the Art), takich jak GPT, Gemini czy Claude, ale odciąć im dostęp do prawdy.

Na tym polega koncepcja, którą wdrożymy w tej lekcji:
1.  **Wykrywamy** dane wrażliwe u siebie (Local Pre-processing).
2.  **Podmieniamy** je na tokeny (np. `Jan Kowalski` -> `<PERSON_1>`).
3.  **Wysyłamy** do modelu tylko strukturę problemu i tokeny.
4.  **Odbieramy** odpowiedź i "tłumaczymy" tokeny z powrotem.

Model dostaje zagadkę logiczną bez danych osobowych. My dostajemy rozwiązanie problemu biznesowego.

### ⚠️ Ważna uwaga prawna (Disclaimer)
Jestem inżynierem, nie prawnikiem. To, co tutaj budujemy, to rozwiązanie techniczne minimalizujące ryzyko wycieku danych. Czy w świetle RODO pseudo-anonimizacja jest wystarczająca dla Twojego konkretnego przypadku (np. danych medycznych)? To pytanie musisz zadać działowi prawnemu. Technologia to narzędzie, ale compliance to proces.

My zajmiemy się stroną techniczną – jak to zbudować, żeby działało skutecznie (ale czy to wystarczy, to już trzeba zbadać z prawnikami pod konkretny przypadek).


Tu pojawia się inżynierski kompromis: **Pseudo-anonimizacja**.

Zwróć uwagę na przedrostek "Pseudo". To kluczowe.
- **Anonimizacja** jest jak niszczarka do papieru. Dane znikają bezpowrotnie. Dla AI to często problem, bo zdanie "XXX spotkał się z YYY w ZZZ" traci sens semantyczny.
- **Pseudo-anonimizacja** jest jak sejf. Zamieniamy "Jan Kowalski" na `<PERSON_1>`, a klucz do tego przypisania trzymamy bezpiecznie u siebie. Wysyłamy do LLM zaszyfrowaną wersję. Model analizuje relacje między `<PERSON_1>` a `<PERSON_2>`, zwraca wynik, a my u siebie podmieniamy to z powrotem.

 Wykorzystamy do tego bibliotekę **Microsoft Presidio**, która częściowa staje się standardem w tym obszarze.

## Wczytujemy biblioteki

Będziemy używać `presidio-analyzer` (do wykrywania) i `presidio-anonymizer` (do ukrywania). Pod spodem Presidio domyślnie korzysta ze **spaCy** do rozumienia kontekstu językowego.

Zainstaluj wymagane pakiety (jeśli jeszcze ich nie masz):

In [ ]:
import spacy
from spacy import displacy
from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern, RecognizerResult, EntityRecognizer
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig
import uuid

## Konfiguracja silnika analizy 
Pod spodem ładuje model spaCy. W produkcji warto załadować to raz (singleton), bo model waży sporo.

In [ ]:
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

## Wykrywanie danych (Analyzer)

Pierwszym krokiem jest identyfikacja. Nie chcemy pisać ręcznie regexów na "każdy możliwy numer telefonu na świecie". Presidio ma wbudowane recognizery dla wielu języków i typów danych (karty kredytowe, krypto-portfele, adresy email, IP, numery telefonów).

Zdefiniujmy tekst, który jest "koszmarem bezpiecznika" – pełen danych osobowych.

In [ ]:
text_to_anonymize = """
Cześć, tu Jan Kowalski. Mój numer to 555-0199-222. 
Proszę o zwrot pieniędzy na kartę 4444 5555 6666 7777. 
Piszcie na jan.kowalski@example.com lub biuro@software30.com.
Mieszkam w Warszawie przy ulicy Złotej 44.
"""

## Analiza tekstu
language='en' używamy tu dla przykładu modelu, ale Presidio wspiera też inne.
Dla polskiego najlepiej wytrenować własny model spaCy lub użyć wielojęzycznego.
Tutaj zasymulujemy dane, które model 'en' zrozumie (numery, maile są uniwersalne).

In [ ]:
analyzer_results = analyzer.analyze(
    text=text_to_anonymize,
    entities=["PHONE_NUMBER", "EMAIL_ADDRESS", "PERSON", "CREDIT_CARD", "LOCATION"],
    language='en' 
)

# Sprawdźmy co znalazł (surowe dane)
for res in analyzer_results:
    print(f"🎯 Wykryto: {res.entity_type} (pewność: {res.score:.2f}) -> '{text_to_anonymize[res.start:res.end]}'")


### ☝️ Co tu się wydarzyło?
Presidio zwróciło listę obiektów `RecognizerResult`. Każdy z nich ma:
- `entity_type`: Co to jest? (np. PHONE_NUMBER)
- `start`/`end`: Gdzie to jest w tekście?
- `score`: Jak bardzo model jest pewny?

To jest fundament. Zauważ, że nie musieliśmy pisać ani jednego wyrażenia regularnego.


### Diagnoza (Czyli dlaczego "out-of-the-box" to za mało)

Uruchomiliśmy kod na domyślnych ustawieniach (`language='en'`, domyślny model). Spójrzmy prawdzie w oczy – wynik, który widzisz powyżej, w systemie produkcyjnym byłby... raczej słabiutki, prawda?

Przeanalizujmy te błędy (tzw. *Failure Modes*), bo one uczą nas najwięcej o naturze LLM i NLP:

1.  **Halucynacje językowe:**
    - `🎯 Wykryto: LOCATION -> 'Cześć'`
    - **Dlaczego?** Model angielski widzi słowo z dużej litery na początku zdania. W jego statystycznym świecie to często nazwa własna (np. miasto). Nie zna słowa "Cześć", więc zgaduje.
2.  **Szum statystyczny:**
    - `🎯 Wykryto: PERSON -> 'na kartę 4444...'`
    - **Dlaczego?** Model widzi ciąg słów i cyfr, którego nie rozumie. Wagi w sieci neuronowej "przechyliły się" w stronę osoby. To typowy false positive.
3.  **Brak Karty Kredytowej:**
    -**Dlaczego?** Presidio ma wbudowany bezpiecznik: sprawdza sumę kontrolną (algorytm Luhna). Testowe numery typu `4444 5555...` są matematycznie niepoprawne, więc system je ignoruje, zakładając, że to przypadkowy ciąg liczb.
4.  **Niska pewność telefonu (0.40):** Dlaczego? Domyślne reguły są bardzo konserwatywne. Jeśli numer nie ma formatu `(555) 123-4567` (USA) lub nie stoi obok słowa "Phone:", system obniża pewność.



## Zobaczmy to na oczy

Jako inżynierowie musimy widzieć, co się dzieje. Czytanie suchych logów tekstowych przy audycie tysięcy dokumentów to nie zawsze wystarcza. Wykorzystamy teraz `displacy` z biblioteki **spaCy**. To standard branżowy do wizualizacji encji nazwanych (NER). Ponieważ Presidio zwraca własny format wyników, napiszemy funkcję pomocniczą `render_ui`, która "przetłumaczy" wyniki Presidio na format zrozumiały dla wizualizera.

Zdefiniujemy też własną paletę kolorów, aby od razu widzieć, co jest czym.

### Funkcja `render_ui`

In [ ]:
def render_ui(results, title=""):
    ents = [{"start": res.start, "end": res.end, "label": res.entity_type} for res in results]
    doc_dict = {"text": text_to_anonymize, "ents": ents, "title": title}

    colors = {
        "POLISH_ADDRESS": "#ffcc00", # Żółty
        "PHONE_NUMBER": "#85C1E9",   # Niebieski
        "EMAIL_ADDRESS": "#D7BDE2",  # Fiolet
        "PERSON": "#ff6961",         # Czerwony
        "LOCATION": "#82E0AA",       # Zielony
        "CREDIT_CARD": "#F7DC6F"     # Złoty
    }

    print("Wyniki detekcji:")
    for res in results:
        print(f" -> [{res.entity_type}] '{text_to_anonymize[res.start:res.end]}' (Score: {res.score:.2f})")

    options = {"ents": list(colors.keys()), "colors": colors}
    return displacy.render(doc_dict, style="ent", manual=True, options=options, jupyter=True)


render_ui(analyzer_results)

### Co widzisz na ekranie?

Spójrz na wygenerowany wynik. To jest ten moment, w którym błędy "krzyczą":

- Widzisz **"Cześć"** podświetlone na zielono jako `LOCATION`?
- Widzisz **wielki czerwony blok `PERSON`** obejmujący słowa "Mieszkam w Warszawie przy"? To oznacza, że zamiast anonimizować dane, system "wycina" logiczną część zdania, czyniąc je niezrozumiałym dla LLM-a.
- Karta kredytowa nie świeci się wcale (wyciek danych!).

Ten widok to nasz punkt wyjścia. Teraz zakasujemy rękawy i naprawiamy to zgodnie ze sztuką inżynierską.


Przejdźmy do naprawy.

## Jak to naprawić? 

Skoro widzimy, że "magia" domyślnych ustawień nie zadziałała, musimy przejąć stery. Pytanie brzmi: jakie mamy opcje i na co realnie mamy wpływ?

W świecie NLP (Natural Language Processing) mamy zazwyczaj dwie drogi naprawy jakości **NER (Named Entity Recognition)**:

#### Opcja 1: "The Hard Way" – Dotrenowanie modelu 
Biblioteka spaCy umożliwia nam wzięcie gotowego modelu i douczenie go na naszych specyficznych danych. Możemy pokazać mu 1000 przykładów naszych faktur czy umów i powiedzieć: *"Patrz, tutaj zawsze występuje ID klienta"*.
- **Plusy:** Najwyższa możliwa precyzja dla specyficznych domen.
- **Minusy:** Wymaga przygotowania zbioru danych (labeling), trwa długo i istnieje ryzyko, że "psujemy" przy okazji to, co model już umiał (tzw. *Catastrophic Forgetting*).
- **Decyzja:** To temat rzeka, który omawiamy w osobnym kursie NLP. Dzisiaj szukamy rozwiązania, które daje 80% efektu przy 20% wysiłku.

#### Opcja 2: Podejście Hybrydowe (Model + Regex)
To jest droga, którą wybierzemy. Połączymy inteligencję gotowego modelu z precyzją reguł.

1.  **Zmiana silnika na polski (`pl_core_news_lg`):** To "low hanging fruit". Po prostu użyjmy modelu, który widział polską odmianę przez przypadki (fleksję) podczas treningu. To powinno wyeliminować halucynacje typu "Cześć = Miasto".
2.  **Wyrażenia regularne (Regex):** Kiedy model AI zawodzi, stary, dobry Regex często ratuje sytuację.
    - *Uważaj jednak:* Regexy są potężne, ale zdradliwe. Łatwo napisać regex, który łapie wszystko, i równie łatwo taki, który staje się nieutrzymywalnym "spaghetti code".
    - *Zastosowanie:* Użyjemy ich tam, gdzie struktura jest sztywna (numery telefonów, specyficzne formaty adresów), a model "gubi się" przez formatowanie.

### Jaki mamy plan dalej?
1.  **Wymienimy model:** Zmienimy silnik na polski (`pl_core_news_lg`), żeby system rozumiał kontekst ("ulica", "mieszkam").
2.  **Napiszemy adapter (`PolishSpacyRecognizer`):** To będzie nasz "tłumacz". Polskie modele spaCy często zwracają techniczne etykiety (np. `persName`), których Presidio nie rozumie. Ten adapter to naprawi w locie.
3.  **Dodamy "sztywne reguły" (Regex):** Dla adresów i telefonów. Gramatyka potrafi zmylić model, ale Regex na `ulic\w*` jest nieugięty i zadziała niezależnie od kontekstu.

Przejdźmy do kodu. Oto kompletna konfiguracja, która zamienia naszą "zabawkę" w narzędzie gotowe na produkcję.

###  Debug: Co widzi sam spaCy? (Analiza surowa)

Zanim zaczniemy pisać kod naprawczy, musimy zrozumieć źródło problemu. Presidio to tylko "wrapper" (opakowanie). Prawdziwą pracę analityczną wykonuje pod spodem silnik `spaCy` i jego model `pl_core_news_lg`.

Zdejmijmy warstwę Presidio i zapytajmy model bezpośrednio: "Co ty tu właściwie widzisz?".

In [ ]:
# Ładujemy model bezpośrednio
nlp = spacy.load("pl_core_news_lg")
doc = nlp(text_to_anonymize)

print(f"surowe widzenie modelu spaCy:")
for ent in doc.ents:
    print(f" - Tekst: '{ent.text}' -> Etykieta: '{ent.label_}'")

### Wnioski

Patrząc na te logi, wszystko staje się jasne. Mamy tu klasyczny problem integracji (Interface Mismatch).

1.  **Dobra wiadomość:** Model jest inteligentny! Wykrył "Jana Kowalskiego" (mimo kropki obok) i "Warszawie" (rozumie polską deklinację). Czyli to nie wina modelu, że nie wykrywaliśmy osób.
2.  **Zła wiadomość:** Model mówi "innym dialektem".
    - Amerykańskie narzędzia (jak Presidio) oczekują standardowych etykiet: `PERSON`, `LOCATION`, `ORGANIZATION`.
    - Polski model zwraca etykiety naukowe (z korpusu NKJP): `persName`, `placeName`, `geogName`.

Dla Presidio etykieta `persName` jest niezrozumiała, więc traktuje ją jak błąd i **ignoruje**. To dlatego Jan Kowalski znikał z wyników.

**Plan naprawczy:** 
Nie musimy trenować nowego modelu. Musimy napisać klasę-adapter (`PolishSpacyRecognizer`), która będzie działać jak tłumacz w locie.
- Gdy spaCy krzyknie: *"Znalazłem `persName`!"*...
- ...nasz adapter przechwyci to i powie do Presidio: *"On ma na myśli `PERSON`"*.

Wdróżmy to teraz w kodzie.

In [ ]:
# 1. KONFIGURACJA BAZOWA
# ----------------------
# Mówimy Presidio: "Użyjemy spaCy z polskim modelem".
configuration = {
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "pl", "model_name": "pl_core_news_lg"}]
}
provider = NlpEngineProvider(nlp_configuration=configuration)
nlp_engine = provider.create_engine()

# 2. KLASA NAPRAWCZA (Custom Recognizer)
# ---------------------------------------
# To jest serce naszego rozwiązania. Klasa, która ręcznie tłumaczy 
# to, co widzi polski model spaCy, na język zrozumiały dla Presidio.
class PolishSpacyRecognizer(EntityRecognizer):
    def __init__(self, supported_language="pl"):
        super().__init__(
            supported_entities=["PERSON", "LOCATION", "ORGANIZATION"], 
            supported_language=supported_language
        )
        self.nlp = spacy.load("pl_core_news_lg")
        
        # PANCERNA MAPA (Wszystkie warianty wielkości liter z korpusu NKJP)
        self.mapping = {
            "persName": "PERSON", "persname": "PERSON", "person": "PERSON", "PER": "PERSON",
            "placeName": "LOCATION", "placename": "LOCATION", "location": "LOCATION", "LOC": "LOCATION",
            "orgName": "ORGANIZATION", "orgname": "ORGANIZATION", "organization": "ORGANIZATION", "ORG": "ORGANIZATION"
        }

    def load(self):
        pass # Model ładowany w __init__

    def analyze(self, text, entities, nlp_artifacts=None):
        results = []
        doc = self.nlp(text)
        for ent in doc.ents:
            if ent.label_ in self.mapping:
                # Tłumaczymy etykietę i zwracamy wynik z wysoką pewnością
                presidio_entity = self.mapping[ent.label_]
                result = RecognizerResult(
                    entity_type=presidio_entity,
                    start=ent.start_char,
                    end=ent.end_char,
                    score=0.85 
                )
                results.append(result)
        return results

# 3. SKŁADANIE SILNIKA I REGEXY
# -----------------------------
analyzer = AnalyzerEngine(nlp_engine=nlp_engine, supported_languages=["pl"])

# A. Dodajemy nasz "Naprawiacz" spaCy
analyzer.registry.add_recognizer(PolishSpacyRecognizer())

# B. ADRES (Polski, odporny na odmianę)
# Łapie: ul., ulica, ulicy, aleja, plac... + Nazwa + Numer
address_regex = r"(?i)(ul\.|ulic\w*|al\.|alej\w*|pl\.|plac\w*)\s+\w+\s+\d+[a-zA-Z]?"
analyzer.registry.add_recognizer(PatternRecognizer(
    supported_entity="POLISH_ADDRESS", 
    patterns=[Pattern("pl_address", address_regex, 0.95)], 
    supported_language="pl"
))

# C. TELEFON (Rozszerzony)
# Łapie też formaty nietypowe (np. 555-0199...)
broad_phone_regex = r"(?:\+?48)?[\s.-]?\d{3,4}[\s.-]?\d{3,4}[\s.-]?\d{3,4}"
analyzer.registry.add_recognizer(PatternRecognizer(
    supported_entity="PHONE_NUMBER", 
    patterns=[Pattern("broad_phone", broad_phone_regex, 0.75)], 
    supported_language="pl"
))

# D. KARTA KREDYTOWA (Słaba walidacja)
# Łapie 4 bloki po 4 cyfry, nawet jeśli to fejkowy numer (bez sumy Luhna)
weak_cc_regex = r"\b\d{4}[ -]?\d{4}[ -]?\d{4}[ -]?\d{4}\b"
analyzer.registry.add_recognizer(PatternRecognizer(
    supported_entity="CREDIT_CARD", 
    patterns=[Pattern("weak_cc", weak_cc_regex, 0.6)], 
    supported_language="pl"
))


## Sprawdzamy

Teraz uruchommy ten sam tekst na nowym silniku. Zwróć uwagę na różnicę w jakości.

In [ ]:
results_pl = analyzer.analyze(
    text=text_to_anonymize,
    entities=["POLISH_ADDRESS", "PHONE_NUMBER", "EMAIL_ADDRESS", "PERSON", "LOCATION", "CREDIT_CARD"],
    language='pl' 
)


render_ui(results_pl, title="Audyt Danych (Silnik PL + Custom Recognizers)")

Jest dobrze? Przeanalizujmy to razem:

1.  **"Cześć"** przestało być miastem (brak halucynacji).
2.  **Jan Kowalski** jest poprawnie oznaczony jako `PERSON` (dzięki naszej klasie).
3.  **Karta kredytowa** została wykryta (dzięki regexowi `weak_cc`).
4.  **Adres** "ulicy Złotej 44" został złapany precyzyjnie (dzięki regexowi na odmianę).

To jest różnica między "uruchomieniem kodu z tutoriala" a "zbudowaniem działającego systemu".

## Pseudo-anonimizacja (maskowanie)

Teraz najważniejsza część. Mamy wykryte miejsca. Musimy je ukryć.
Mamy trzy główne strategie (Operators):

1.  **Mask (Redact):** Zastąpienie znakami `***`. Dobre do logów, złe dla LLM (traci strukturę).
2.  **Replace:** Zastąpienie tokenem `<ENTITY>`. Standardowe podejście. LLM wie, że "coś tu było".
3.  **Hash/Encrypt:** Odwracalna kryptografia. Pozwala na pełny "round-trip".

Zastosujmy podejście **Replace**, które jest najbardziej czytelne dla modeli językowych.

In [ ]:
# Konfiguracja operatorów - co robimy z każdym typem danych?
operators = {
    # Fallback dla nieznanych typów
    "DEFAULT": OperatorConfig("replace", {"new_value": "<SECRET>"}), 
    
    # Zachowujemy typ danych w tokenie (Contextual AI)
    "PHONE_NUMBER": OperatorConfig("replace", {"new_value": "<PHONE_ID>"}),
    "PERSON": OperatorConfig("replace", {"new_value": "<PERSON_ID>"}),
    "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "<EMAIL_ID>"}),
    
    # Nasze niestandardowe encje i miasta
    "POLISH_ADDRESS": OperatorConfig("replace", {"new_value": "<ADDRESS_ID>"}),
    "LOCATION": OperatorConfig("replace", {"new_value": "<CITY_ID>"}),
    
    # Karty kredytowe maskujemy, zostawiając końcówkę (np. do weryfikacji przez usera)
    "CREDIT_CARD": OperatorConfig("mask", {
        "masking_char": "*", 
        "chars_to_mask": 12, 
        "from_end": False
    }),
}

# Wykonanie anonimizacji na wynikach z naszego poprawionego silnika (results_pl)
# UWAGA: Używamy tutaj zmiennej 'results_pl' z poprzedniego kroku (gdzie użyliśmy polskiego silnika)
anonymized_result = anonymizer.anonymize(
    text=text_to_anonymize,
    analyzer_results=results_pl, 
    operators=operators
)

print("\n🔒 TEKST PO ANONIMIZACJI (Bezpieczny dla LLM):")
print("-" * 60)
print(anonymized_result.text)
print("-" * 60)

# Sprawdźmy JSON-a, którego moglibyśmy wysłać do aplikacji frontendowej
print("\n Raport przekształceń (Items):")
for item in anonymized_result.items:
    print(f" - Zamieniono {item.entity_type} na: {item.text}")

### Wynik 
Powinno się pojawić coś w stylu:
`Cześć, tu <PERSON_ID>. Mój numer to <PHONE_ID>. Proszę o zwrot pieniędzy na kartę ************7777...`

Zauważ różnicę w traktowaniu karty kredytowej. Użyliśmy operatora `mask`, zostawiając ostatnie 4 cyfry. Dlaczego? Bo to często wystarcza agentowi AI, żeby zidentyfikować kartę w systemie bankowym ("Czy chodzi o kartę kończącą się na 7777?"), a jednocześnie pełny numer nie wycieka. To jest **Contextual AI** w praktyce.


In [ ]:
class StatefulAnonymizer:
    def __init__(self):
        self.mapping = {} # Pamięć: { "<PERSON_1>": "Jan Kowalski" }
        self.counter = {} # Licznik: { "PERSON": 1 }

    def get_token(self, entity_type, original_text):
        # Sprawdź czy już znamy tę konkretną wartość (np. ten sam email występuje 2 razy)
        # W prostym demo kluczem jest wartość tekstowa.
        # W produkcji: warto normalizować (np. lower case dla maili).
        
        # Szukamy w istniejących mapowaniach (reverse lookup)
        for token, original in self.mapping.items():
            if original == original_text and token.startswith(f"<{entity_type}"):
                return token
        
        # Jeśli nie ma, tworzymy nowy
        if entity_type not in self.counter:
            self.counter[entity_type] = 1
        else:
            self.counter[entity_type] += 1
            
        token = f"<{entity_type}_{self.counter[entity_type]}>"
        self.mapping[token] = original_text
        return token

    def anonymize(self, text, analyzer_results):
        # Sortujemy wyniki od końca, żeby zmiana indeksów nie zepsuła tekstu
        sorted_results = sorted(analyzer_results, key=lambda x: x.start, reverse=True)
        new_text = list(text)
        
        for res in sorted_results:
            original_value = text[res.start:res.end]
            token = self.get_token(res.entity_type, original_value)
            
            # Podmiana w tekście (slice assignment)
            new_text[res.start:res.end] = list(token)
            
        return "".join(new_text)

    def deanonymize(self, text):
        # Prosta pętla po mapie. 
        # UWAGA: W produkcji lepiej użyć regexa dla bezpieczeństwa, żeby nie podmienić podciągów.
        result = text
        for token, original in self.mapping.items():
            result = result.replace(token, original)
        return result

## Testujemy

In [ ]:
vault = StatefulAnonymizer()
complex_text = "Jan Kowalski rozmawia z Anną Nowak. Potem Jan Kowalski dzwoni do Anny."

## Analiza

In [ ]:
results = analyzer.analyze(
    text=complex_text, 
    entities=["PERSON"], 
    language='pl'
)

## Anonimizacja z pamięcią

In [ ]:
safe_text = vault.anonymize(complex_text, results)

print(f"1. Oryginał:  {complex_text}")
print(f"2. Safe Text: {safe_text}")
print(f"3. Mapa:      {vault.mapping}")

# Symulacja: LLM generuje odpowiedź używając tokenów
llm_response = "Zadanie przydzielono do <PERSON_1>, ponieważ <PERSON_2> jest na urlopie."
print(f"4. LLM mówi:  {llm_response}")

# 3. Deanonimizacja (powrót)
final_text = vault.deanonymize(llm_response)
print(f"5. Finalnie:  {final_text}")

### Analiza wyniku
Zobaczysz, że system poprawnie rozpoznał, że "Jan Kowalski" (występujący 2 razy) to ta sama osoba i przypisał jej jeden token `<PERSON_1>`, a "Anna Nowak" dostała `<PERSON_2>`.
Kiedy LLM zwrócił odpowiedź używając tokenów, my bez problemu podmieniliśmy je z powrotem na nazwiska.

To jest właśnie istota bezpiecznego przetwarzania danych w Software 3.0. Agent jest "stateless" (widzi tylko tokeny), a my trzymamy "state" (mapę) w naszej bezpiecznej infrastrukturze.

## Podsumowanie

Co dokładnie się tu wydarzyło i jak masz to interpretować? Wyciągnijmy najważniejsze wnioski:

1.  **Modele zamknięte robią świetną robotę.** Bądźmy szczerzy – kompetencje inżynierskie firm takich jak OpenAI czy Anthropic są trudne do skopiowania. Zamiast z nimi walczyć, warto skorzystać z ich mocy obliczeniowej, używając "nakładki bezpieczeństwa", którą właśnie zbudowaliśmy.
2.  **Mit lokalnych modeli.** Ponad 99% firm nie potrzebuje modeli lokalnych w konfiguracji "odciętej od świata". Próba budowania własnej, prywatnej chmury często kończy się fiaskiem finansowym i inżynierskim. To ogromny wysiłek utrzymaniowy, który rzadko się zwraca.
3.  **Siła modeli "lekkich".** Zauważ, że całą "brudną robotę" (wykrywanie danych) zrobiliśmy lokalnie na CPU, używając spaCy. Modele z rodziny spaCy są szybkie, dają się łatwo "podkręcać" (tuning) pod nasze potrzeby i nie wymagają drogich GPU.
4.  **Architektura hybrydowa.** To jest złoty środek. Lokalny, szybki model (Total Local) do wykrywania i sanityzacji danych, a następnie bezpieczny transfer zanonimizowanej "esencji" problemu do potężnego modelu w chmurze.

Dzięki temu podejściu masz ciastko (bezpieczeństwo danych) i jesz ciastko (inteligencja modelu SOTA).

## Przydatne linki
- [Microsoft Presidio Documentation](https://microsoft.github.io/presidio/)
- [Spacy EntityRecognizer](https://spacy.io/api/entityrecognizer)
- [spaCy Models for Polish](https://spacy.io/models/pl)
- [Preventing PII leakage when using LLMs: An introduction to Microsoft’s Presidio](https://ploomber.io/blog/presidio/)
- Publikacja [Detection of Personal Data in Structured Datasets Using a Large Language Model](https://arxiv.org/pdf/2506.22305)
- Publikacja [Adaptive PII Mitigation Framework for Large Language Models](https://arxiv.org/pdf/2501.12465)
- Publikacja [AudAgent: Automated Auditing of Privacy Policy Compliance in AI Agents](https://arxiv.org/pdf/2511.07441)